In [3]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [25]:
class Batsmen(TypedDict):

    runs: int
    balls: int
    four : int
    six: int

    sr : float
    bpb:float
    bp: float
    summary: str

In [26]:
graph = StateGraph(Batsmen)

In [27]:
def calculate_sr(state: Batsmen):
    runs = state['runs']
    balls = state['balls']

    state['sr'] = runs / balls

    return {'sr':state['sr']}

In [28]:
def calculate_bpb(state : Batsmen):
    balls = state['balls']
    four = state['four']
    six = state['six']

    state['bpb'] = balls // (four+six)

    return {'bpb' : state['bpb']}

In [29]:
def calculate_bp(state : Batsmen):
    runs = state['runs']
    four = state['four']
    six = state['six']

    state['bp'] = (four * 4 + six * 6) * 100 / runs

    return {'bp':state['bp']}


In [30]:
def generate_summary(state:Batsmen):
    summary = f'''
    Strike Rate = {state['sr']}
    Balls Per Boundary = {state['bpb']}
    Boundary Percentage = {state['bp']}
    '''

    state['summary'] = summary

    return state

In [31]:
# add nodes

graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bp',calculate_bp)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('generate_summary',generate_summary)

In [32]:
#  add edges -> Important for Parallel Execution

graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bp')
graph.add_edge(START,'calculate_bpb')

graph.add_edge('calculate_sr','generate_summary')
graph.add_edge('calculate_bp','generate_summary')
graph.add_edge('calculate_bpb','generate_summary')

graph.add_edge('generate_summary',END)

In [33]:
workflow = graph.compile()

In [34]:
initial_state = {'runs':125,'balls':93,'four':7,'six':3}

final_state = workflow.invoke(initial_state)

print(final_state)

{'runs': 125, 'balls': 93, 'four': 7, 'six': 3, 'sr': 1.3440860215053763, 'bpb': 9, 'bp': 36.8, 'summary': '\n    Strike Rate = 1.3440860215053763\n    Balls Per Boundary = 9\n    Boundary Percentage = 36.8\n    '}
